In [1]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from datasets import Dataset
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, accuracy_score, f1_score
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments,
    EvalPrediction
)

# Model
MODEL_NAME = "indolem/indobertweet-base-uncased" 
print(f"Menggunakan model: {MODEL_NAME}")

# Load Data
print("Memuat dataset latih dan uji...")
df_train = pd.read_csv('../resources/train_preprocessing_typo_correction.csv')
df_val = pd.read_csv('../resources/val_preprocessing_typo_correction.csv')
df_test = pd.read_csv('../resources/test_preprocessing_typo_correction.csv')

df_train = df_train.dropna(subset=['Pesan_Clean', 'Label Toxic'])
df_val = df_val.dropna(subset=['Pesan_Clean', 'Label Toxic'])
df_test = df_test.dropna(subset=['Pesan_Clean', 'Label Toxic'])

# Mapping Label
label_mapping = {"normal": 0, "toxic": 1}
for df in [df_train, df_val, df_test]:
    df['Label_Num'] = df['Label Toxic'].map(label_mapping)

X_train, y_train = df_train['Pesan_Clean'].astype(str).tolist(), df_train['Label_Num'].tolist()
X_val, y_val = df_val['Pesan_Clean'].astype(str).tolist(), df_val['Label_Num'].tolist()
X_test, y_test = df_test['Pesan_Clean'].astype(str).tolist(), df_test['Label_Num'].tolist()

# Class Weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"Device yang digunakan: {device}")
print(f"Bobot Kelas (Normal, Toxic): {class_weights}")

# Tokenization
print("Memulai tokenisasi teks...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_data(texts, labels):
    encodings = tokenizer(texts, truncation=True, padding='max_length', max_length=64)
    dataset = Dataset.from_dict({
        'input_ids': encodings['input_ids'],
        'attention_mask': encodings['attention_mask'],
        'labels': labels
    })
    return dataset

train_dataset = tokenize_data(X_train, y_train)
val_dataset = tokenize_data(X_val, y_val)
test_dataset = tokenize_data(X_test, y_test)

# Model Initialization
print("Mengunduh dan menyiapkan arsitektur IndoBERTweet...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=2,
    id2label={0: "normal", 1: "toxic"},
    label2id={"normal": 0, "toxic": 1}
).to(device)

# Trainer
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(p: EvalPrediction):
    preds = np.argmax(p.predictions, axis=1)
    
    acc = accuracy_score(p.label_ids, preds)
    f1_macro = f1_score(p.label_ids, preds, average='macro')
    
    return {
        "accuracy": acc,
        "f1_macro": f1_macro
    }

# Training Arguments
training_args = TrainingArguments(
    output_dir='./results_toxic',
    num_train_epochs=5,              
    per_device_train_batch_size=32,  
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=100,
    weight_decay=0.01,               
    eval_strategy="epoch",           
    save_strategy="epoch",
    logging_dir='./logs_toxic',
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True
    # fp16=True
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# Training
print("\n🚀 Mulai proses Fine-Tuning IndoBERTweet...")
trainer.train()

# Evaluation
# print("\nMelakukan prediksi pada data Testing...")
# predictions = trainer.predict(test_dataset)
# y_pred = np.argmax(predictions.predictions, axis=-1)

# print("\n=== HASIL KLASIFIKASI INDOBERTweet (TOXIC) ===")
# print(classification_report(y_test, y_pred, target_names=["normal", "toxic"]))

print("\nMelakukan prediksi pada data Validation...")
val_predictions = trainer.predict(val_dataset)
y_val_pred = np.argmax(val_predictions.predictions, axis=-1)

print("\n=== HASIL KLASIFIKASI INDOBERTweet VALIDATION (TOXIC) ===")
print(classification_report(y_val, y_val_pred, target_names=["normal", "toxic"]))


print("\nMelakukan prediksi pada data Testing...")
test_predictions = trainer.predict(test_dataset)
y_test_pred = np.argmax(test_predictions.predictions, axis=-1)

print("\n=== HASIL KLASIFIKASI INDOBERTweet TESTING (TOXIC) ===")
print(classification_report(y_test, y_test_pred, target_names=["normal", "toxic"]))

# Save
print("\nMenyimpan model final...")
trainer.save_model("./model_indobertweet_chat_toxic_preprocessing_typo_correction")
tokenizer.save_pretrained("./model_indobertweet_chat_toxic_preprocessing_typo_correction")
print("Selesai! Model berhasil disimpan.")

[Output tidak disertakan pada rilis publik. Hanya statistik agregat yang dipublikasikan; teks pesan pengguna, daftar frekuensi kata, dan identitas pengguna dihilangkan. Lihat 'Data & Content Policy' di README.]


Menggunakan model: indolem/indobertweet-base-uncased
Memuat dataset latih dan uji...
Device yang digunakan: cuda
Bobot Kelas (Normal, Toxic): [ 0.52108688 12.35571365]
Memulai tokenisasi teks...


[Output tidak disertakan pada rilis publik. Hanya statistik agregat yang dipublikasikan; teks pesan pengguna, daftar frekuensi kata, dan identitas pengguna dihilangkan. Lihat 'Data & Content Policy' di README.]


Mengunduh dan menyiapkan arsitektur IndoBERTweet...


[Output tidak disertakan pada rilis publik. Hanya statistik agregat yang dipublikasikan; teks pesan pengguna, daftar frekuensi kata, dan identitas pengguna dihilangkan. Lihat 'Data & Content Policy' di README.]


...


=== HASIL KLASIFIKASI INDOBERTweet VALIDATION (TOXIC) ===
              precision    recall  f1-score   support

      normal       0.96      0.99      0.97      9820
       toxic       0.81      0.63      0.71       996

    accuracy                           0.95     10816
   macro avg       0.89      0.81      0.84     10816
weighted avg       0.95      0.95      0.95     10816


Melakukan prediksi pada data Testing...


[Output tidak disertakan pada rilis publik. Hanya statistik agregat yang dipublikasikan; teks pesan pengguna, daftar frekuensi kata, dan identitas pengguna dihilangkan. Lihat 'Data & Content Policy' di README.]



=== HASIL KLASIFIKASI INDOBERTweet TESTING (TOXIC) ===
              precision    recall  f1-score   support

      normal       0.99      1.00      0.99      9296
       toxic       0.78      0.55      0.64       280

    accuracy                           0.98      9576
   macro avg       0.88      0.77      0.82      9576
weighted avg       0.98      0.98      0.98      9576


Menyimpan model final...


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.15it/s]

Selesai! Model berhasil disimpan.


In [2]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from datasets import Dataset
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, accuracy_score, f1_score
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments,
    EvalPrediction
)

# Model
MODEL_NAME = "indolem/indobertweet-base-uncased" 
print(f"Menggunakan model: {MODEL_NAME}")

# Load Data
print("Memuat dataset latih dan uji...")
df_train = pd.read_csv('../resources/train_preprocessing_typo_correction.csv')
df_val = pd.read_csv('../resources/val_preprocessing_typo_correction.csv')
df_test = pd.read_csv('../resources/test_preprocessing_typo_correction.csv')

df_train = df_train.dropna(subset=['Pesan_Clean', 'Label Judi Online'])
df_val = df_val.dropna(subset=['Pesan_Clean', 'Label Judi Online'])
df_test = df_test.dropna(subset=['Pesan_Clean', 'Label Judi Online'])

# Mapping Label
label_mapping = {"normal": 0, "judi online": 1}
for df in [df_train, df_val, df_test]:
    df['Label_Num'] = df['Label Judi Online'].map(label_mapping)

X_train, y_train = df_train['Pesan_Clean'].astype(str).tolist(), df_train['Label_Num'].tolist()
X_val, y_val = df_val['Pesan_Clean'].astype(str).tolist(), df_val['Label_Num'].tolist()
X_test, y_test = df_test['Pesan_Clean'].astype(str).tolist(), df_test['Label_Num'].tolist()

# Class Weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"Device yang digunakan: {device}")
print(f"Bobot Kelas (Normal, Judi Online): {class_weights}")

# Tokenization
print("Memulai tokenisasi teks...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_data(texts, labels):
    encodings = tokenizer(texts, truncation=True, padding='max_length', max_length=64)
    dataset = Dataset.from_dict({
        'input_ids': encodings['input_ids'],
        'attention_mask': encodings['attention_mask'],
        'labels': labels
    })
    return dataset

train_dataset = tokenize_data(X_train, y_train)
val_dataset = tokenize_data(X_val, y_val)
test_dataset = tokenize_data(X_test, y_test)

# Model Initialization
print("Mengunduh dan menyiapkan arsitektur IndoBERTweet...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=2,
    id2label={0: "normal", 1: "judi online"},
    label2id={"normal": 0, "judi online": 1}
).to(device)

# Trainer
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(p: EvalPrediction):
    preds = np.argmax(p.predictions, axis=1)
    
    acc = accuracy_score(p.label_ids, preds)
    f1_macro = f1_score(p.label_ids, preds, average='macro')
    
    return {
        "accuracy": acc,
        "f1_macro": f1_macro
    }

# Training Arguments
training_args = TrainingArguments(
    output_dir='./results_toxic',
    num_train_epochs=5,              
    per_device_train_batch_size=32,  
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=100,
    weight_decay=0.01,               
    eval_strategy="epoch",           
    save_strategy="epoch",
    logging_dir='./logs_toxic',
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True
    # fp16=True
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# Training
print("\n🚀 Mulai proses Fine-Tuning IndoBERTweet...")
trainer.train()

# Evaluation
# print("\nMelakukan prediksi pada data Testing...")
# predictions = trainer.predict(test_dataset)
# y_pred = np.argmax(predictions.predictions, axis=-1)

# print("\n=== HASIL KLASIFIKASI INDOBERTweet (JUDI ONLINE) ===")
# print(classification_report(y_test, y_pred, target_names=["normal", "judi online"]))

print("\nMelakukan prediksi pada data Validation...")
val_predictions = trainer.predict(val_dataset)
y_val_pred = np.argmax(val_predictions.predictions, axis=-1)

print("\n=== HASIL KLASIFIKASI INDOBERTweet VALIDATION (JUDI ONLINE) ===")
print(classification_report(y_val, y_val_pred, target_names=["normal", "judi online"]))


print("\nMelakukan prediksi pada data Testing...")
test_predictions = trainer.predict(test_dataset)
y_test_pred = np.argmax(test_predictions.predictions, axis=-1)

print("\n=== HASIL KLASIFIKASI INDOBERTweet TESTING (JUDI ONLINE) ===")
print(classification_report(y_test, y_test_pred, target_names=["normal", "judi online"]))

# Save
print("\nMenyimpan model final...")
trainer.save_model("./model_indoberttweet_chat_judi_online_preprocessing_typo_correction")
tokenizer.save_pretrained("./model_indoberttweet_chat_judi_online_preprocessing_typo_correction")
print("Selesai! Model berhasil disimpan.")

Menggunakan model: indolem/indobertweet-base-uncased
Memuat dataset latih dan uji...
Device yang digunakan: cuda
Bobot Kelas (Normal, Judi Online): [0.53041669 8.71917163]
Memulai tokenisasi teks...
Mengunduh dan menyiapkan arsitektur IndoBERTweet...


[Output tidak disertakan pada rilis publik. Hanya statistik agregat yang dipublikasikan; teks pesan pengguna, daftar frekuensi kata, dan identitas pengguna dihilangkan. Lihat 'Data & Content Policy' di README.]



🚀 Mulai proses Fine-Tuning IndoBERTweet...


[Output tidak disertakan pada rilis publik. Hanya statistik agregat yang dipublikasikan; teks pesan pengguna, daftar frekuensi kata, dan identitas pengguna dihilangkan. Lihat 'Data & Content Policy' di README.]


[Output tidak disertakan pada rilis publik. Hanya statistik agregat yang dipublikasikan; teks pesan pengguna, daftar frekuensi kata, dan identitas pengguna dihilangkan. Lihat 'Data & Content Policy' di README.]


...


=== HASIL KLASIFIKASI INDOBERTweet VALIDATION (JUDI ONLINE) ===
              precision    recall  f1-score   support

      normal       0.99      0.99      0.99     10702
 judi online       0.25      0.43      0.31       114

    accuracy                           0.98     10816
   macro avg       0.62      0.71      0.65     10816
weighted avg       0.99      0.98      0.98     10816


Melakukan prediksi pada data Testing...


[Output tidak disertakan pada rilis publik. Hanya statistik agregat yang dipublikasikan; teks pesan pengguna, daftar frekuensi kata, dan identitas pengguna dihilangkan. Lihat 'Data & Content Policy' di README.]



=== HASIL KLASIFIKASI INDOBERTweet TESTING (JUDI ONLINE) ===
              precision    recall  f1-score   support

      normal       1.00      0.99      0.99      9481
 judi online       0.45      0.60      0.51        95

    accuracy                           0.99      9576
   macro avg       0.72      0.80      0.75      9576
weighted avg       0.99      0.99      0.99      9576


Menyimpan model final...


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.03it/s]

Selesai! Model berhasil disimpan.
